In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

import shapely
from shapely.geometry import Point

In [2]:
import folium
from folium.plugins import PolyLineTextPath

from branca.element import Element, Template
import json

In [3]:
def get_matched_length(cmp, xd, cmp_segid, xd_segid):
    ls1 = cmp.loc[cmp['cmp_segid'].eq(cmp_segid)].to_crs('epsg:2227').iloc[0]['geometry'].geoms[0]
    ls2 = xd.loc[xd['XDSegID'].eq(xd_segid)].to_crs('epsg:2227').iloc[0]['geometry']
    a1 = Point(ls1.coords[0])
    b1 = Point(ls1.coords[-1])
    return abs(ls2.project(a1)-ls2.project(b1))

In [4]:
## TEST VERSION 2 (CHANGE OF THE ICONS IS DYNAMIC, BUT IS BUILT BASED ON EACH SINGLE SEGMENT OF ROUTE)
def plot_map(cmp_segid, segments, cmp_shp, xd_shp):
    seg = segments.loc[segments['cmp_segid'].eq(cmp_segid)]
    xd_shp = xd_shp.loc[xd_shp['XDSegID'].isin(seg['inrix_segid'])].to_crs('epsg:4326')

    xd_shp = pd.merge(xd_shp, seg[['inrix_segid','old','new','length_matched_new']],
                      left_on='XDSegID', right_on='inrix_segid')
    
    cmp_shp = cmp_shp.loc[cmp_shp['cmp_segid'].eq(cmp_segid)].to_crs('epsg:4326')
    
    
    
    if seg.empty:
        print(f"No matched_path_gdf features for trip {trip_id_to_plot}")
        return folium.Map()  # empty base map
    
    # -- Create map --
    center = xd_shp.geometry.unary_union.centroid.coords[0]
    m = folium.Map(location=[center[1], center[0]], zoom_start=15, tiles="cartodbpositron")
    
    for _, row in xd_shp.iterrows():
        color = "#007AFF"
        weight = 5
        if row['old'] == 0 and row['new'] == 1:
            color = "#32fbe0"
            weight = 10
        if row['old'] == 1 and row['new'] == 0:
            color = "#eb233e"
            weight = 10
        coords = [(pt[1], pt[0]) for pt in row.geometry.coords]
    
        # 1) Draw the base polyline
        # 1a) Add a popup to each segment showing its sequence (rownum) and OSMID
        popup = folium.Popup(
            f"XDSegID: {row['XDSegID']}<br>Length Matched: {row['length_matched_new']}",
            max_width=200, sticky = True
        )
        tooltip = folium.Tooltip(
            f"XDSegID: {row['XDSegID']}<br>Length Matched: {row['length_matched_new']}",
            sticky = True
        )
        popup.options.update({
            "autoClose": False,
            "closeOnClick": True
        })
        
        poly = folium.PolyLine(
            locations=coords,
            color=color,
            opacity=0.8,
            popup = popup,
            tooltip = tooltip,
            weight = weight,
        ).add_to(m)

        # 2) Add ▶ arrowheads (and "=") along the line, letting Leaflet.TextPath auto-rotate ▶ 
        arrow_layer = PolyLineTextPath(
            poly,
            text="=▶",
            repeat="50%",      # percent‐based spacing (initial)
            offset=15,          # pixels above the centerline
            orientation="auto",# auto-rotate the ▶ glyph along the segment
            attributes={
                "fill": "#969696",
                "font-size": "12px",
                "font-weight": "bold",
            }
        ).add_to(m)
    
    color = "#eb6b34"
    coords = []
    for geom in cmp_shp.iloc[0].geometry.geoms:
        for pt in geom.coords:
            coords.append((pt[1], pt[0]))
    cmp_poly = folium.PolyLine(
        locations=coords,
        color=color,
        opacity=0.8,
        #popup = popup
    ).add_to(m)


    # -- Display map --
    return m

In [5]:
#OLD = r'Q:\CMP\LOS Monitoring 2022\Network_Conflation\v2202\conflation_script_test\CMP_Segment_INRIX_Links_Correspondence_2202_Manual.csv'
#NEW = r'Q:\CMP\LOS Monitoring 2025\Network_Conflation\v2501\CMP_Segment_INRIX_Links_Correspondence_2501_Manual-expandednetwork.csv'

#OLD = r'Q:/CMP/LOS Monitoring 2023/Network_Conflation/v2301/CMP_Segment_INRIX_Links_Correspondence_2301.csv'
#NEW = r'Q:\Data\Observed\Streets\INRIX\v2501\network_conflation\CMP\CMP_Segment_INRIX_Links_Correspondence_2501_Manual.csv'

OLD = r'Q:\CMP\LOS Monitoring 2024\Network_Conflation\v2401\CMP_Segment_INRIX_Links_Correspondence_2401_Manual-expandednetwork.csv'
NEW = r'Q:\CMP\LOS Monitoring 2024\Network_Conflation\v2401\CMP_Segment_INRIX_Links_Correspondence_2401_Manual.csv'

In [6]:
#XD_OLD = r'Q:/GIS/Transportation/Roads/INRIX/XD/2202/INRIX_XD-SF-2202.gpkg'

#XD_OLD = 'Q:/GIS/Transportation/Roads/INRIX/XD/2301/INRIX_XD-SF-2301.gpkg'
#XD_NEW = 'Q:/GIS/Transportation/Roads/INRIX/XD/2501/INRIX_XD-SF-2501.gpkg'

XD_OLD = 'Q:/GIS/Transportation/Roads/INRIX/XD/2401/INRIX_XD-SF-2401.gpkg'
XD_NEW = 'Q:/GIS/Transportation/Roads/INRIX/XD/2401/INRIX_XD-SF-2401.gpkg'

CMP = r'Q:\GIS\Transportation\Roads\CMP\cmp_roadway_segments-expanded-v202204.gpkg'

In [7]:
rv2301 = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2301\maprelease-xdreplaced\USA_California.csv')
rv2301.rename(columns={'XDId_23_1':'new','XDId_22_1':'old'}, inplace=True)
rv2301['version'] = 2301
rv2401 = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2401\maprelease-xdreplaced\USA_California.csv')
rv2401.rename(columns={'XDId_24_1':'new','XDId_23_1':'old'}, inplace=True)
rv2401['version'] = 2401
rv2501 = pd.read_csv(r'Q:\GIS\Transportation\Roads\INRIX\XD\2501\maprelease-xdreplaced\USA_California.csv')
rv2501.rename(columns={'XDId_25_1':'new','XDId_24_1':'old'}, inplace=True)
rv2501['version'] = 2501

In [8]:
rep = pd.concat([rv2301, rv2401, rv2501])

In [9]:
old = pd.read_csv(OLD)
old.rename(columns={c:c.lower() for c in old.columns}, inplace=True)
old['length_matched'] = old['length_matched'].round(2)
new = pd.read_csv(NEW)
new.rename(columns={c:c.lower() for c in new.columns}, inplace=True)
new['length_matched'] = new['length_matched'].round(2)

In [10]:
xd_old = gpd.read_file(XD_OLD)
xd_new = gpd.read_file(XD_NEW)

In [11]:
xd_old['XDSegID'] = xd_old['XDSegID'].astype('int64')

In [12]:
cmp = gpd.read_file(CMP)

In [13]:
old['old'] = 1
new['new'] = 1

In [14]:
df = pd.merge(old, 
              new, 
              on=['cmp_segid','inrix_segid'], 
              how='outer',
              suffixes=['_old','_new']).fillna(0)

In [15]:
df.loc[df['cmp_segid'].eq(1),['length_matched_old','length_matched_new']].sum()

length_matched_old    2534.00
length_matched_new    2533.23
dtype: float64

In [16]:
s = df.groupby('cmp_segid').agg({'length_matched_old':'sum',
                                 'length_matched_new':'sum'})

In [17]:
added_xd_iter =  s.loc[(s['length_matched_old'] - s['length_matched_new']).gt(10)].iterrows()
#iter(df.groupby('cmp_segid')) #iter(df.loc[df['old'].eq(0)].groupby('cmp_segid'))

In [18]:
s.loc[(s['length_matched_old'] - s['length_matched_new']).gt(10)].loc[:245]

,length_matched_old,length_matched_new
cmp_segid,,
10,5020.36,3149.62
31,15254.23,15097.06
32,8210.82,8197.14
70,2570.05,1990.55
111,10793.35,10132.45
113,10752.55,10111.00
123,3545.41,3430.83
132,2720.13,2702.99
133,4187.31,3623.82


In [56]:
cmp_segid = next(added_xd_iter)[0]
#cmp_segid = 29
l0, l1 = round(s.loc[cmp_segid,'length_matched_old'], 0), round(s.loc[cmp_segid,'length_matched_new'], 0)
print('CMP Seg ID: {}\nOld Length Matched: {}\nNew Length Matched: {}\nDifference: {}\nPct Diff: {}'.format(
        cmp_segid, l0, l1, l1 - l0, (l1 - l0) / l0
        )
     )

CMP Seg ID: 246
Old Length Matched: 3608.0
New Length Matched: 0.0
Difference: -3608.0
Pct Diff: -1.0


In [57]:
plot_map(cmp_segid, df, cmp, xd_new)

In [21]:
plot_map(cmp_segid, df, cmp, xd_old)

In [22]:
df.loc[df['cmp_segid'].eq(cmp_segid)]

,cmp_segid,inrix_segid,length_matched_old,old,length_matched_new,new
147,10,170081991,222.89,1.0,226.23,1.0
148,10,170663170,580.69,1.0,580.69,1.0
149,10,429475281,624.87,1.0,0.00,0.0
150,10,449826929,630.67,1.0,0.00,0.0
151,10,449826930,626.17,1.0,626.17,1.0
152,10,449826931,352.43,1.0,361.92,1.0
153,10,485562815,605.05,1.0,605.05,1.0
154,10,485565285,628.03,1.0,0.00,0.0
155,10,485591185,651.02,1.0,651.02,1.0
156,10,1626746166,98.54,1.0,98.54,1.0


In [23]:
get_matched_length(cmp, xd_new, cmp_segid,  399693150

)

0.0

In [24]:
'''
Unavoidable inconsistency:
    23
    30
    100
    138
    141
    236

Improvement inconsistency:
    27
    40
    58
    70
    78
    81
    86
    90
    108
    109
    111
    113
    115
    123
    139 (potentially significant change)
    140 (potentially significant change)
    152
    192
    193
    194
    203
    206
    225
    

Marginal inconsistency
    74 (consider manual drop of 449826687)
    75 (consider manual drop of 449826694)
    77 (consider manual drop of 1626662946)
'''

'\nUnavoidable inconsistency:\n    23\n    30\n    100\n    138\n    141\n    236\n\nImprovement inconsistency:\n    27\n    40\n    58\n    70\n    78\n    81\n    86\n    90\n    108\n    109\n    111\n    113\n    115\n    123\n    139 (potentially significant change)\n    140 (potentially significant change)\n    152\n    192\n    193\n    194\n    203\n    206\n    225\n    \n\nMarginal inconsistency\n    74 (consider manual drop of 449826687)\n    75 (consider manual drop of 449826694)\n    77 (consider manual drop of 1626662946)\n'

In [25]:
'''
cmp_segid 23, confirmed merge: (1626738758,  1626764660, 1626764515) -> 937045027
cmp_segid 27, confirmed replace: 429465693 -> 937040647
    TODO: manual add 1626734566
cmp_segid 30, confirmed replace: 1626683868 -> 937077505
cmp_segid 34, confirmed gap between San Jose Ave and Rhine due to 1-way.  
    confirmed replace: 1626690046 -> 476203781 (prior to v2501)
cmp_segid 35, confirmed replace: 1626612264 -> (485581613, 449826607)
cmp_segid 36, confirmed replace: 1626612245 -> (485577541, 449826616)
cmp_segid 38, confirmed replace: 449830832 -> (476205518, 476188677)
cmp_segid 39, confirmed replace: 449826759 -> (476195817, 476188556)
cmp_segid 43, confirmed replace: 400266828 -> (476175702, 476194142)
cmp_segid 44, confirmed replace: 399841892 -> (476211512, 476178019)
cmp_segid 46, confirmed replace: 1626626524 -> 476189585
cmp_segid 53, confirmed replace: 1626626347 -> 476174221
cmp_segid 59, confirmed replace: 1626653300 -> 476208541
cmp_segid 63, confirmed replace: 441585708 -> (937010064, 937075397)
'''

'\ncmp_segid 23, confirmed merge: (1626738758,  1626764660, 1626764515) -> 937045027\ncmp_segid 27, confirmed replace: 429465693 -> 937040647\n    TODO: manual add 1626734566\ncmp_segid 30, confirmed replace: 1626683868 -> 937077505\ncmp_segid 34, confirmed gap between San Jose Ave and Rhine due to 1-way.  \n    confirmed replace: 1626690046 -> 476203781 (prior to v2501)\ncmp_segid 35, confirmed replace: 1626612264 -> (485581613, 449826607)\ncmp_segid 36, confirmed replace: 1626612245 -> (485577541, 449826616)\ncmp_segid 38, confirmed replace: 449830832 -> (476205518, 476188677)\ncmp_segid 39, confirmed replace: 449826759 -> (476195817, 476188556)\ncmp_segid 43, confirmed replace: 400266828 -> (476175702, 476194142)\ncmp_segid 44, confirmed replace: 399841892 -> (476211512, 476178019)\ncmp_segid 46, confirmed replace: 1626626524 -> 476189585\ncmp_segid 53, confirmed replace: 1626626347 -> 476174221\ncmp_segid 59, confirmed replace: 1626653300 -> 476208541\ncmp_segid 63, confirmed repla